# 03 Load Database

Load the cleaned tables from `data/processed/` into the local PostgreSQL database `nyc_restaurants` (port 5433).

In [11]:
%pip install sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


### Connect to the database

In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

PROCESSED_DIR = "../data/processed/"
DB_URL = "postgresql+psycopg2://localhost:5433/nyc_restaurants"

engine = create_engine(DB_URL)

# Connection test
with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

PostgreSQL 18.6 (Postgres.app) on aarch64-apple-darwin23.6.0, compiled by Apple clang version 15.0.0 (clang-1500.3.9.4), 64-bit


### Read cleaned tables with explicit data types
CSV files carry no type information. Codes like `"04"` and census tracts like `"000700"` must be read as text to keep leading zeros.

In [13]:
RESTAURANT_TEXT_COLS = [
    "dba", "boro", "building", "street", "zipcode", "phone",
    "cuisine_description", "community_board", "council_district", "census_tract", "nta",
]

restaurants = pd.read_csv(
    PROCESSED_DIR + "restaurants.csv",
    dtype={c: str for c in RESTAURANT_TEXT_COLS},
)

inspections = pd.read_csv(
    PROCESSED_DIR + "inspections.csv",
    parse_dates=["inspection_date", "grade_date"],
)
inspections["score"] = inspections["score"].astype("Int64")

violations = pd.read_csv(
    PROCESSED_DIR + "violations.csv",
    dtype={"violation_code": str, "violation_category": str},
)

# Quick check: text codes keep leading zeros
print(violations["violation_category"].value_counts().head())
print(restaurants[["zipcode", "census_tract"]].head())

violation_category
10    76428
06    54060
04    51111
02    37782
08    31732
Name: count, dtype: int64
  zipcode census_tract
0   11229       055200
1   10007       002100
2   10018       008400
3   11215       014100
4   11224       034800


### Recreate tables from the schema file
Re-running `sql/01_create_tables.sql` makes this notebook safe to run repeatedly.

In [14]:
# Recreate empty tables from the schema file, so this notebook can be re-run safely
with open("../sql/01_create_tables.sql") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.exec_driver_sql(schema_sql)

print("Tables recreated.")

Tables recreated.


### Load tables
Parent tables first (restaurants → inspections → violations) because of foreign keys. `if_exists="append"` keeps the schema and constraints defined in SQL.

In [15]:
def load_table(df, table_name, engine):
    """Append a DataFrame into an existing table whose schema is defined in SQL."""
    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=5000,
    )
    print(f"{table_name:<12} loaded {len(df):>7} rows")


# Order matters: parent tables first, because of foreign keys
load_table(restaurants, "restaurants", engine)
load_table(inspections, "inspections", engine)
load_table(violations, "violations", engine)

restaurants  loaded   31369 rows
inspections  loaded   92713 rows
violations   loaded  287206 rows


In [16]:
# Verify: row counts in the database should match the DataFrames
with engine.connect() as conn:
    for name, df in [("restaurants", restaurants), ("inspections", inspections), ("violations", violations)]:
        n_db = conn.execute(text(f"SELECT COUNT(*) FROM {name};")).scalar()
        status = "OK" if n_db == len(df) else "CHECK"
        print(f"{name:<12} csv: {len(df):>7}   database: {n_db:>7}   {status}")

restaurants  csv:   31369   database:   31369   OK
inspections  csv:   92713   database:   92713   OK
violations   csv:  287206   database:  287206   OK


### Validate in SQL
Reconcile counts across tables, check for orphan records, and run a first analytical query.

In [17]:
def q(sql):
    """Run a SQL query and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

In [18]:
q("""
SELECT
    (SELECT SUM(n_violations) FROM inspections)                         AS sum_n_violations,
    (SELECT COUNT(*) FROM violations)                                   AS violation_rows,
    (SELECT SUM(n_critical) FROM inspections)                           AS sum_n_critical,
    (SELECT COUNT(*) FROM violations WHERE critical_flag = 'Critical')  AS critical_rows;
""")

,sum_n_violations,violation_rows,sum_n_critical,critical_rows
0,287206,287206,153861,153861


In [19]:
q("""
SELECT COUNT(*) AS orphan_inspections
FROM inspections AS i
LEFT JOIN restaurants AS r
    ON i.camis = r.camis
WHERE r.camis IS NULL;
""")

,orphan_inspections
0,0


In [20]:
q("""
SELECT
    r.boro,
    COUNT(*)                    AS n_inspections,
    ROUND(AVG(i.score), 1)      AS avg_score,
    ROUND(AVG(i.n_critical), 2) AS avg_critical_violations
FROM inspections AS i
JOIN restaurants AS r
    ON i.camis = r.camis
WHERE i.in_scope
  AND i.inspection_stage = 'Initial Inspection'
GROUP BY r.boro
ORDER BY avg_score DESC;
""")

,boro,n_inspections,avg_score,avg_critical_violations
0,Queens,13134,19.7,2.11
1,Brooklyn,14736,18.6,1.93
2,Bronx,5249,18.1,1.92
3,Manhattan,21660,17.8,1.88
4,Staten Island,2043,17.8,1.97
5,None,50,15.5,1.60


In [21]:
q("""
SELECT
    r.boro,
    ROUND(AVG(i.score), 1)                                AS avg_score,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY i.score)  AS median_score
FROM inspections AS i
JOIN restaurants AS r ON i.camis = r.camis
WHERE i.in_scope
  AND i.inspection_stage = 'Initial Inspection'
GROUP BY r.boro
ORDER BY avg_score DESC;
""")

,boro,avg_score,median_score
0,Queens,19.7,13.0
1,Brooklyn,18.6,13.0
2,Bronx,18.1,13.0
3,Manhattan,17.8,12.0
4,Staten Island,17.8,13.0
5,None,15.5,12.0
